<a href="https://colab.research.google.com/github/williamlidberg/Analyses-of-Environmental-Data-2/blob/main/modules/module_5/Assignment_5_2026.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Geopandas 2026

You can think of GeoPandas as pandas for vector data. By vector data, I mean points, lines, and polygons where each feature has a corresponding row in an attribute table. This is also referred to as tabular data. You have used GeoPandas in previous modules, but the aim of this module is to prepare you for the next module, which covers machine learning on vector data.

Start by installing GeoPandas and a few other packages.

In [ ]:
!pip install geopandas
!pip install matplotlib # for plotting
!pip install folium # for data exploration
!pip install mapclassify # for data exploration

# Reading and writing data
The best thing about GeoPandas is that it can handle multiple file types such as GeoPackage, GeoJSON, and Shapefiles. Shapefiles can only contain about 4 GB of data, and since many modern datasets are much larger than that, the industry is slowly moving over to GeoPackage. You can easily read all these data types with the GeoPandas command `read_file()`.

Start by downloading some data to work with.

### Download ditches

In [ ]:
from urllib.request import urlretrieve
url = ('https://geodata.naturvardsverket.se/nedladdning/diken/Diken_Sverige/Diken_lansvis/Diken_K.zip')
filename = '/content/Diken_K.zip' # you need to adjust this path on your own computer if you are using anaconda.
urlretrieve(url, filename)
!unzip -o /content/Diken_K.zip -d /content/ditches

### Download wetlands

In [ ]:
from urllib.request import urlretrieve
url = ('https://geodata.naturvardsverket.se/nedladdning/VMI/ursprunglig_digitalisering/K_Blekinge_VMI.zip')
filename = '/content/K_Blekinge_VMI.zip' # you need to adjust this path on your own computer if you are using anaconda.
urlretrieve(url, filename)
!unzip -o '/content/K_Blekinge_VMI.zip' -d /content/wetlands

### Download protected areas

In [ ]:
from urllib.request import urlretrieve
url = ('https://geodata.naturvardsverket.se/nedladdning/naturvardsregistret/biosfarsomraden.zip')
filename = '/content/biosfarsomraden.zip' # you need to adjust this path on your own computer if you are using anaconda.
urlretrieve(url, filename)
!unzip -o '/content/biosfarsomraden.zip' -d /content/protected

### Read all data into GeoDataFrames
All file types can be read with the same command.

In [ ]:
import geopandas as gpd
ditches = gpd.read_file('/content/ditches/Diken_K.gpkg') # Note that this is a geopackage
wetlands = gpd.read_file('/content/wetlands/K_Blekinge_VMI_Ytor.shp')
protected = gpd.read_file('/content/protected/BIOSFARSOMRADEN.shp')

You can also write the GeoDataFrames to different file types. For example, to save the wetlands polygons as a GeoJSON, we can specify the driver as 'GeoJSON'.

In [ ]:
wetlands.to_file("/content/wetlands.geojson", driver="GeoJSON")

## Task 1
a ) Save the protected areas dataframe as a GeoPackage. \
b ) Save the wetland polygons dataframe as a GeoJSON.

# Visualizing data
GeoPandas includes a function to quickly inspect the data if the dataset is not too big. This works for the wetland polygons, but it would probably crash Google Colab if you tried this with the ditch dataframe. This is a very neat command to remember.

In [ ]:
wetlands.explore(
    column="NVKLASS",
    tiles="OpenTopoMap" # ask your nearest LLM for more options
)

# Selecting data by attributes
It's quite common to be interested in only parts of the dataset during an analysis. In most GIS programs, you can run a tool named something along the lines of "select by attributes". Since GeoPandas builds on pandas, this can easily be achieved with Python as well.

Let's pretend that we want to investigate ditches, but only forest ditches. We can then select or slice the dataframe by that attribute. In our case: 'Typ'.

In [ ]:
ditches

In [ ]:
forest_ditches = ditches[ditches["Typ"] == "Skogsdike"] # Skogsdike is Swedish for forest ditch

Sometimes we want to make multiple selections at the same time. This is where the `&` operator can be used. Let's select forest ditches that are longer than 500 meters by adding the length attribute to our selection.

In [ ]:
forest_ditches_500m = ditches[(ditches["Typ"] == "Skogsdike") & (ditches["Langd_m"] > 500)] # Langd, or Längd is Swedish for length
forest_ditches_500m.explore(
    column="Langd_m",
    tiles="Esri.WorldGrayCanvas"
)

### Task 2
Make a selection on the wetlands GeoDataFrame and answer the question: How many wetlands have a high nature value (Högt naturvärde), an area over 10 hectares, and a perimeter less than 2000 meters?

# Spatial relations
One of the most commonly used operations within GIS is to select features from one dataset that falls within the features of another dataset. This is commonly refered to as spatial join or intersect.

Lets select all wetlands that fall within the protective area using geopandas.

In [ ]:
selected_wetlands = gpd.sjoin(wetlands, protected, predicate='within')
selected_wetlands.explore(
    column="NVKLASS",
    tiles="Esri.WorldGrayCanvas"
)

You can reverse this selection by using the `~` operator.

In [ ]:
non_intersecting_wetlands = wetlands[~wetlands.index.isin(selected_wetlands.index)]
non_intersecting_wetlands .explore(column="NVKLASS",
                                   tiles="Esri.WorldImagery")

The spatial join operation has many different options:
*   contains
*   within
*   touches
*   crosses
*   overlaps
*   equals

## Task 3
Use GeoPandas to select ditches that cross the wetlands and display them on a map.

# Math operations
Another common operation is to summarize the area, length, or features that fall within other features. For example, we can summarize the total length of ditches within the wetlands.

In [ ]:
joined_gdf = gpd.sjoin(ditches, wetlands, how="inner", predicate='intersects') # intersect ditches with wetlands
joined_gdf['line_length'] = joined_gdf.geometry.length # calculate lenght of intersected ditches
summary = joined_gdf.groupby('NVKLASS')['line_length'].sum() #summarize the lenght of ditch lines within each wetland (right refers to the right geodataframe in the intersect)

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

sns.barplot(x=summary.index, y=summary.values)
plt.xlabel('NVKLASS')
plt.ylabel('Length of Ditch Lines in meter')
plt.title('Length of Ditch Lines within Wetlands by NVKLASS')
plt.xticks(rotation=90)
plt.show()


BY looking at this plot it seems like more ditches = higher nature values of wetlands. Excellent, lets drain more wetlands!


## Task 4
I am afraid that we failed to take the size of each wetland into account. Calculate the area of each wetland and divide the ditch length by the wetland area and make a new plot with drainage intensity vs nature value "NVKLASS". Hints: use commands like .geometry.area and .geometry.length and then join these results like we did above and sum the lenght/area. (Ask a LLM for hints if you get stuck).

# Buffers
Creating a buffer around features is also a very common tool for geospatial analysis.

In [ ]:
import geopandas as gpd

wetlands_buffer = wetlands.geometry.buffer(500) # buffer wetlands with 500 meters
wetlands_buffer_gdf = gpd.GeoDataFrame(geometry=wetlands_buffer) # create a new geodataframe from the buffered geometries

wetlands_buffer_gdf.explore(
    tiles="Esri.WorldImagery")

If you look at the map, you can see that some polygons overlap each other. Sometimes this is not what you want, and in those cases, they can be merged by dissolving the overlaps.

In [ ]:
import geopandas as gpd

wetlands_buffer = wetlands.buffer(500)
singlepart = wetlands_buffer.unary_union # Dissolve overlapping buffers
singlepart = gpd.GeoDataFrame(geometry=[singlepart])
singlepart.crs = wetlands.crs # set crs to the new data
singlepart.explore(tiles="Esri.WorldImagery")

## Task 5
Let's say we want to restore some drained wetlands, but at the same time, we don't want to destroy any cultural sites nearby. We can help with this by finding drained wetlands that do not have any cultural sites within 300 meters. Download water-related cultural sites from [Länsstyrelsen](https://ext-geodatakatalog.lansstyrelsen.se/GeodataKatalogen/srv/swe/catalog.search#/search?isTemplate=n&referenceDateRange=%7B%22range%22:%7B%22referenceDateRange%22:%7B%22gte%22:null,%22lte%22:null,%22relation%22:%22within%22%7D%7D%7D&metadataDateRange=%7B%22range%22:%7B%22metadataDateRange%22:%7B%22gte%22:null,%22lte%22:null,%22relation%22:%22within%22%7D%7D%7D&metadataAndReferenceDateRange=%7B%22range%22:%7B%22metadataAndReferenceDateRange%22:%7B%22gte%22:null,%22lte%22:null,%22relation%22:%22within%22%7D%7D%7D&sortBy=relevance&sortOrder=&query_string=%7B%22spatialRepresentationType%22:%7B%22vector-point%22:true%7D,%22ownerOrgName%22:%7B%22L%C3%A4nsstyrelsen%20Blekinge%20l%C3%A4n%22:true%7D%7D&from=1&to=30) and select wetlands with some nature value (Vissa naturvärden) that are more than 300 meters away from cultural sites. Save the selected wetlands as a GeoPackage.

In [ ]:
# You can download the data using urllib
from urllib.request import urlretrieve
url = ('https://ext-dokument.lansstyrelsen.se/gemensamt/geodata/ShapeExport/Lstk.Inventerade_vattenanknutna_kulturmiljoer.zip')
filename = '/content/Lstk.Inventerade_vattenanknutna_kulturmiljoer.zip' # you need to adjust this path on your own computer if you are using anaconda.
urlretrieve(url, filename)
!unzip -o '/content/Lstk.Inventerade_vattenanknutna_kulturmiljoer.zip' -d /content/culture

# Data aggregation
There is a global geospatial index based around nested hexagon polygons which is called H3. It was developed by [Uber](https://www.uber.com/en-SE/blog/h3/) and later open-sourced for everyone to use.

* H3 is a hierarchical geospatial index.
* H3 was developed to address the challenges of Uber's data science needs.
* H3 can be used to join disparate datasets.
* In addition to the benefits of the hexagonal grid shape, H3 includes features for modeling flow.
* H3 is well-suited for applying machine learning to geospatial data.

H3 can be very useful if you are working in an area and need to aggregate multiple data sources. Since the polygons have unique IDs and are constant for everyone, you can easily share data with other people based on the H3 index.

Also, [hexagons are the bestagons](https://www.youtube.com/watch?v=thOifuHs6eY).

In [ ]:
!pip install tobler
!pip install h3
!pip install h3pandas

First, we need to select which geographical area we want to work with. Start by downloading some Swedish regions from [SCB](https://www.scb.se/hitta-statistik/regional-statistik-och-kartor/regionala-indelningar/digitala-granser/).

In [ ]:
from urllib.request import urlretrieve
url = ('https://www.scb.se/contentassets/3443fea3fa6640f7a57ea15d9a372d33/shape_svenska_240104.zip')
filename = '/content/Sweden_shapefile.zip' # you need to adjust this path on your own computer if you are using anaconda.
urlretrieve(url, filename)
!unzip -o /content/Sweden_shapefile.zip -d /content/sweden
!unzip -o /content/sweden/LanSweref99TM.zip -d /content/sweden/lan

We can keep working with Blekinge since it is small and we already have some data for it.

In [ ]:
sweden = gpd.read_file('/content/sweden/lan/Lan_Sweref99TM_region.shp')
blekinge = sweden[sweden["LnNamn"] == "Blekinge"] # Skogsdike is Swedish for forest ditch
blekinge.plot()

With this polygon, we can now call the H3 API and create a hexagon index for Blekinge. There are multiple resolutions available, where the largest hexagon is 4,357,449.41 km² and the smallest is 0.895 m².

The resolution parameter ranges from 0 to 15, but due to limited RAM on Colab, we will use 7 here.

In [ ]:
import h3pandas
import tobler
blekinge_hexagons = tobler.util.h3fy(blekinge, resolution=7, clip=False, buffer=False, return_geoms=True)
blekinge_hexagons.explore(tiles="Esri.WorldImagery")

Note the `hex_id` column. Different resolutions have different IDs, but each polygon ID is unique, which means you can use it to join and share data based on this global ID.

In [ ]:
blekinge_hexagons

Let's summarize the number of ditch lines within each hexagon.

In [ ]:
import geopandas as gpd

# make sure that the coordinate system is the same
ditches  = ditches.to_crs(blekinge_hexagons.crs)

# Perform a spatial join
joined = gpd.sjoin(blekinge_hexagons, ditches, predicate='intersects', how='left')

# Group by hex_id and count the number of sites within each hexagon
summary = joined.groupby('hex_id').size().reset_index(name='ditch_count')

# Merge the summary with the hexagon GeoDataFrame to add the cnumber of sites within each hexagon
hexagons_with_sites = blekinge_hexagons.merge(summary, on='hex_id', how='left')
hexagons_with_sites.explore('ditch_count',
                            tiles="Esri.WorldImagery")

However, since some ditches are longer than others, it makes more sense to summarize the length of the ditches instead of the number of line segments.

In [ ]:
import geopandas as gpd

ditches = ditches.to_crs(blekinge_hexagons.crs)
joined = gpd.sjoin(blekinge_hexagons, ditches, predicate='intersects', how='left')

# Calculate the length of each ditch line
joined['line_length'] = joined.geometry.length

# Group by hex_id and sum the length of ditch lines within each hexagon
summary = joined.groupby('hex_id')['line_length'].sum().reset_index()
hexagons_with_ditch_length = blekinge_hexagons.merge(summary, on='hex_id', how='left')
hexagons_with_ditch_length.explore('line_length',
                                   tiles="Esri.WorldImagery")

Finally, we can aggregate more data into the hexagons. For example, let's also add the area of wetlands within each hexagon.

In [ ]:

wetlands = wetlands.to_crs(hexagons_with_ditch_length.crs)
joined_wetlands = gpd.sjoin(hexagons_with_ditch_length, wetlands, predicate='intersects', how='left')

# Calculate the area of each wetland polygon
joined_wetlands['wetland_area'] = joined_wetlands.geometry.area

# Group by hex_id and sum the area of wetlands within each hexagon
summary_wetlands = joined_wetlands.groupby('hex_id')['wetland_area'].sum().reset_index()

hexagons_with_both_ditch_lenght_and_wetland_area = hexagons_with_ditch_length.merge(summary_wetlands, on='hex_id', how='left')
hexagons_with_both_ditch_lenght_and_wetland_area

### Task 6
Add the number of cultural sites to these hexagons so you get ditch length, wetlands area and number of cultural sites within the same geodataframe.

# Joining spatial data with tabular data
Sometimes you have a dataset with an ID column and another dataset with coordinates. A common workflow is to join the table to the spatial data using a common ID column. Here is an example where we download a land use dataset from SMHI and vector polygons of watersheds from SMHI. Start by downloading the tabular data from SMHI: https://vattenwebb.smhi.se/modelregion/. Note that due to decisions made by SMHI, there is no direct URL to the file. Look for "Markanvändning" and "Delavrinningsområdet". You have to manually navigate to the page, click the link to the Excel file, download it, and then upload it to Google Drive. Once downloaded, you can simply drag and drop the file into the file list on the left.

In [ ]:
import pandas as pd

file_path = "/content/Markanvändning-delavrinningsområde.xls"
landUse = pd.read_excel(file_path, sheet_name=1, decimal=",", skiprows=0)  # adjust skiprows if needed
landUse

What is this nonsens? Someone at SMHI decided that it was a good idea to include a bunch of text on the top of the excel sheet! We need to clean this before it can be used.

In [ ]:
import pandas as pd

file_path = "/content/Markanvändning-delavrinningsområde.xls"

# Read the sheet, using row 4 as the header (zero-based index)
landUse = pd.read_excel(
    file_path,
    sheet_name=1,
    header=4,
    decimal=","
)

# Remove completely empty rows and columns
landUse = landUse.dropna(axis=0, how="all")
landUse = landUse.dropna(axis=1, how="all")

# Remove any remaining unnamed columns
landUse = landUse.loc[
    :, ~landUse.columns.astype(str).str.contains("^Unnamed")
]

# Keep only rows with a valid Subid
landUse = landUse[
    pd.to_numeric(landUse["Subid"], errors="coerce").notna()
].copy()

# Convert Subid to integer
landUse["Subid"] = landUse["Subid"].astype(int)

# Clean up the index
landUse = landUse.reset_index(drop=True)

display(landUse)

Download the watershed polygons

In [ ]:
import requests
from zipfile import ZipFile
from io import BytesIO
import geopandas as gpd

url = "https://opendata-download.smhi.se/svar/SVAR2022_delavrinningsomraden.zip"
folder = "SMHI_Avrinningsomraden"

# Download ZIP
response = requests.get(url)
response.raise_for_status()

# Extract ZIP using zipfile instead of unzip
with ZipFile(BytesIO(response.content)) as z:
    z.extractall(folder)



In [ ]:
subWatersheds = gpd.read_file("/content/SMHI_Avrinningsomraden/SVAR2022_delavrinningsomraden.gpkg")
subWatersheds

Perform a spatial join between the two. The columns in the table will be joined to the vector data.

In [ ]:
# Merge land use into polygons
subWatersheds = subWatersheds.merge(
    landUse, # fromt he excel file
    left_on='ARO_UUID',
    right_on='Aroid',
    how='left'  # use 'left' to keep all polygons
)



Let's inspect the harvested area across watersheds in Västernorrland (the area that was recently flooded).

In [ ]:
sweden = gpd.read_file('/content/sweden/lan/Lan_Sweref99TM_region.shp')
Västernorrlands = sweden[sweden["LnNamn"] == "Västernorrlands"] # select Västernorrland


In [ ]:
import matplotlib.pyplot as plt

Västernorrlands_watersheds = subWatersheds[subWatersheds.intersects(Västernorrlands.geometry.union_all())].copy()
Västernorrlands = Västernorrlands.to_crs(subWatersheds.crs)

fig, ax = plt.subplots(figsize=(10, 10))
Västernorrlands.plot(ax=ax, color='none', edgecolor='black', linewidth=1)
Västernorrlands_watersheds.plot(
    ax=ax,
    column='Hygge',
    cmap='YlOrBr',
    legend=True,
    legend_kwds={'label': "Harvested area (% of watershed)", 'shrink': 0.6},
    edgecolor='grey',
    linewidth=0.5
)

ax.set_title("Watersheds intersecting Västernorrlands colored by harvested area")
ax.set_axis_off()
plt.tight_layout()
plt.show()

### Task 7
Visit https://vattenwebb.smhi.se/modelregion/ and select another dataset like "Jordarter" or "Belastning". Join it to the watershed and plot it. Feel free to select another region of Sweden. The page is in Swedish but Jordarter = soils, Belastning = nutrient load from each watershed.


# Conclusions
GeoPandas is a powerful tool to wrangle geospatial vector data and can read and write multiple different file formats. Aggregating data is very useful for further analyses, and we will use this skill in the next module to combine aggregated vector data with machine learning to predict things on the H3 geospatial index. Here is an example of how it can be used in the real world: https://beta.behovskartan.se/